## Imports

In [ ]:
# ============================================================
# 0. IMPORTS
# ============================================================

import os
import re
import gc
import json
import time
from dataclasses import dataclass, asdict, field
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from tqdm.auto import tqdm
from numba import njit, uint32
from dataclasses import dataclass, field

## Config

In [ ]:
# ============================================================
# 1. CONFIG
# ============================================================

@dataclass
class Config:
    # -------------------------
    # Model type
    # -------------------------
    model_type: str = "endogenous"
    # allowed:
    # "fixed"
    # "endogenous"

    # -------------------------
    # Network
    # -------------------------
    N: int = 40000

    network_type: str = "barabasi_albert"
    # allowed:
    # "erdos_renyi"
    # "watts_strogatz"
    # "newman_watts_strogatz"
    # "barabasi_albert"
    # "random_regular"
    # "ring_regular_lattice"
    # "square_lattice"
    # "complete_graph"
    # "well_mixed_sampled"
    
    network_params: dict = field(default_factory=lambda: {
        #"k": 4,
        #"alpha": 0.0,
        # ER: "p": 0.0001
        # WS: "beta": 0.1
        # NWS: "p": 0.0001
        "m": 2,  # BA
        # square_lattice: "periodic": True
        # well_mixed_sampled: "k_payoff": 4,
    })

    ensure_connected: bool = False

    # -------------------------
    # Network randomicity
    # -------------------------
    randomize_network: bool = False
    network_generations: int = 1
    runs_per_network: int = 10

    # -------------------------
    # Updating rule and updating mode
    # -------------------------
    update_rule: str = "fermi"
    # allowed:
    # "fermi"
    # "proportional_imitation"

    update_mode: str = "async"
    # allowed:
    # "async"
    # "sync"

    omega: float = 0.2  # used only by Fermi

    # -------------------------
    # Dynamics
    # -------------------------
    extra_sweeps: int = 400
    record_every: int = 1
    seed: int = 0

    # -------------------------
    # Initial conditions
    # -------------------------
    rho0_values: tuple = (0.5,)

    # -------------------------
    # Payoff mode
    # -------------------------
    normalize_payoff: bool = False
    # True  = average payoff
    # False = accumulated / total payoff
        # -------------------------
    # Self-interaction
    # -------------------------
    self_interaction: bool = False
    # False = standard case, no self-interaction
    # True  = each node also plays against itself when computing payoff
    # -------------------------
    # Fixed-game scan
    # -------------------------
    grid_points: int = 21   # numero di punti per asse

    fixed_game_pairs: tuple = field(default_factory=lambda: ())

    # fixed_game_pairs can also be manually specified, for example:
    #
    # fixed_game_pairs: tuple = (
    #     (-0.5, 0.5),
    # )

    # -------------------------
    # Endogenous feedback
    # -------------------------
    feedback: str = "rho"
    # allowed:
    # "rho"
    # "rho2"

    G1_grid_points: int = 21

    G1_pairs: tuple = field(default_factory=lambda: ())

    D2_g: float = 1.0
    D2_r: float = 1.0

    def __post_init__(self):
        # Fixed-game grid, only if needed
        if not self.fixed_game_pairs:
            grid = np.linspace(-1.0, 1.0, self.grid_points)

            self.fixed_game_pairs = tuple(
                (round(Dg, 2), round(Dr, 2))
                for Dg in grid
                for Dr in grid
            )

        # ---- SH region only ----
        if not self.G1_pairs:
            Dg_vals = np.linspace(-1.0, 0.0, 11)   # 11 points
            Dr_vals = np.linspace(0.0, 1.0, 11)    # 11 points

            self.G1_pairs = tuple(
                (round(Dg, 2), round(Dr, 2))
                for Dg in Dg_vals
                for Dr in Dr_vals
                if not (
                    round(Dg, 2) == self.D2_g
                    and round(Dr, 2) == self.D2_r
                )
            )

    # -------------------------
    # Parallelism
    # -------------------------
    max_workers: int = 1

    # -------------------------
    # Early stopping
    # -------------------------
    early_stop_enabled: bool = False

    # Stop immediately if all-C or all-D is reached
    early_stop_absorbing: bool = False

    # Do not check stabilization before this number of sweeps
    early_stop_min_sweeps: int = 100

    # Window, in sweeps, over which rho must be stable
    early_stop_window: int = 200

    # Check stabilization every X sweeps
    early_stop_check_every: int = 50

    # Maximum allowed variation of rho inside the window
    early_stop_rho_tol: float = 5e-3

    # Maximum allowed difference between first half and second half of the window
    early_stop_slope_tol: float = 1e-3

    # -------------------------
    # Checkpoints
    # -------------------------
    save_checkpoints: bool = True
    checkpoint_every: int = 200
    checkpoint_mode: str = "overwrite"
    # allowed:
    # "overwrite"
    # "history"

    save_final_checkpoint: bool = True

    # -------------------------
    # Snapshots
    # -------------------------
    save_snapshots: bool = False
    snap_every: int = 100
    save_initial_snapshot: bool = False
    save_final_snapshot: bool = False

    # -------------------------
    # Network visualisation
    # -------------------------
    save_network_visualization: bool = False
    snapshot_layout_seed: int = 0
    snapshot_node_size: float = 20.0
    snapshot_edge_width: float = 0.25
    snapshot_edge_alpha: float = 0.25

    # -------------------------
    # Output roots
    # -------------------------
    out_root: str = "results"
    fig_root: str = "figures"
    chk_root: str = "checkpoints"
    snap_root: str = "snapshots"
    net_root: str = "saved_networks"

cfg = Config(
    # -------------------------
    # Model type
    # -------------------------
    model_type="endogenous",

    # -------------------------
    # Network: sampled well-mixed population
    # -------------------------
    N=1000,
    network_type="well_mixed_sampled",
    network_params={
        "k_payoff": 4,
    },

    ensure_connected=False,

    # -------------------------
    # Network randomicity
    # -------------------------
    randomize_network=False,
    network_generations=1,
    runs_per_network=10,

    # -------------------------
    # Updating rule and updating mode
    # -------------------------
    update_rule="fermi",
    update_mode="async",
    omega=0.2,

    # -------------------------
    # Dynamics
    # -------------------------
    # New simulation: run directly for 5000 sweeps.
    # Since G2 is different, results go into a different folder.
    extra_sweeps=0,
    record_every=1,
    seed=0,

    # -------------------------
    # Initial conditions
    # -------------------------
    rho0_values=(0.1, 0.5, 0.9),

    # -------------------------
    # Payoff mode
    # -------------------------
    normalize_payoff=False,

    # -------------------------
    # Endogenous feedback
    # -------------------------
    feedback="rho",

    # G2 = Snowdrift Game
    D2_g=1.0,
    D2_r=-1.0,

    # -------------------------
    # Selected G1 games
    # -------------------------
    G1_pairs=(
        (-0.6, -0.4),
        (-0.7, 0.3),
        (-0.3, 0.7),
        (0.6, 0.4),
    ),

    # -------------------------
    # Parallelism
    # -------------------------
    max_workers=1,

    # -------------------------
    # Early stopping
    # -------------------------
    early_stop_enabled=False,
    early_stop_absorbing=False,

    # -------------------------
    # Checkpoints
    # -------------------------
    save_checkpoints=True,
    checkpoint_every=200,
    checkpoint_mode="overwrite",
    save_final_checkpoint=True,

    # -------------------------
    # Snapshots
    # -------------------------
    save_snapshots=False,
    save_initial_snapshot=False,
    save_final_snapshot=False,

    # -------------------------
    # Network visualisation
    # -------------------------
    save_network_visualization=False,

    # -------------------------
    # Output roots
    # -------------------------
    out_root="results",
    fig_root="figures",
    chk_root="checkpoints",
    snap_root="snapshots",
    net_root="saved_networks",
)

## Tags and Paths

In [ ]:
# ============================================================
# 2. TAGS AND PATHS
# ============================================================

def safe_tag(x):
    return str(x).replace(".", "p").replace("-", "m").replace(" ", "").replace("'", "").replace('"', "")

def omega_tag(cfg):
    return f"omega{safe_tag(float(cfg.omega))}"

def self_interaction_tag(cfg):
    if getattr(cfg, "self_interaction", False):
        return "self"
    return None

def model_tag(cfg):
    if cfg.model_type == "fixed":
        return "fixed"
    if cfg.model_type == "endogenous":
        return "endo"
    raise ValueError("model_type must be 'fixed' or 'endogenous'")


def update_rule_tag(cfg):
    if cfg.update_rule == "fermi":
        return "upd_fermi"
    if cfg.update_rule == "proportional_imitation":
        return "upd_propimit"
    raise ValueError("update_rule must be 'fermi' or 'proportional_imitation'")


def update_mode_tag(cfg):
    if cfg.update_mode == "async":
        return "mode_async"
    if cfg.update_mode == "sync":
        return "mode_sync"
    raise ValueError("update_mode must be 'async' or 'sync'")


def feedback_tag(cfg):
    if cfg.model_type == "fixed":
        return None
    if cfg.feedback == "rho":
        return "fb_rho"
    if cfg.feedback == "rho2":
        return "fb_rho2"
    raise ValueError("feedback must be 'rho' or 'rho2'")


def payoff_mode_tag(cfg):
    return "avg" if cfg.normalize_payoff else "tot"


def network_randomicity_tag(cfg):
    if cfg.randomize_network:
        return f"netrand_{int(cfg.network_generations)}"
    return "netrand_0"


def network_tag(cfg):
    nt = cfg.network_type.lower()
    p = cfg.network_params

    name_map = {
        "erdos_renyi": "er",
        "watts_strogatz": "ws",
        "newman_watts_strogatz": "nws",
        "barabasi_albert": "ba",
        "random_regular": "rr",
        "ring_regular_lattice": "ring",
        "square_lattice": "sq",
        "complete_graph": "complete",
        "well_mixed_sampled": "wmix",
    }

    if nt not in name_map:
        raise ValueError("Unsupported network_type.")

    parts = [f"net_{name_map[nt]}", f"N{cfg.N}"]

    if nt == "erdos_renyi":
        parts.append(f"p{safe_tag(p['p'])}")

    elif nt == "watts_strogatz":
        parts += [f"k{p['k']}", f"b{safe_tag(p['beta'])}"]

    elif nt == "newman_watts_strogatz":
        parts += [f"k{p['k']}", f"p{safe_tag(p['p'])}"]

    elif nt == "barabasi_albert":
        parts.append(f"m{p['m']}")

    elif nt == "random_regular":
        parts.append(f"k{p['k']}")

    elif nt == "ring_regular_lattice":
        parts += [f"k{p['k']}", f"a{safe_tag(float(p.get('alpha', 0.0)))}"]

    elif nt == "square_lattice":
        neigh = p.get("neighborhood", "von_neumann").lower()

        if neigh in ["von_neumann", "vn"]:
            # Backward-compatible: keep the old folder name
            parts += [
                f"a{safe_tag(float(p.get('alpha', 0.0)))}",
                f"per{int(bool(p.get('periodic', True)))}",
            ]

        elif neigh == "moore":
            # New folder only for Moore neighbourhood
            parts += [
                "moore",
                f"a{safe_tag(float(p.get('alpha', 0.0)))}",
                f"per{int(bool(p.get('periodic', True)))}",
            ]

        else:
            raise ValueError("neighborhood must be 'von_neumann' or 'moore'.")
        
    elif nt == "complete_graph":
        pass
    
    elif nt == "well_mixed_sampled":
        parts.append(f"kpay{int(p.get('k_payoff', 4))}")

    if cfg.ensure_connected:
        parts.append("gc")

    # Add omega directly to the network tag.
    # This prevents simulations with different selection intensities
    # from being saved in the same path.
    parts.append(omega_tag(cfg))

    return "_".join(parts)


def fixed_game_tag(D_g, D_r):
    return f"G_Dg{safe_tag(D_g)}_Dr{safe_tag(D_r)}"


def g2_tag(cfg):
    return f"G2_Dg{safe_tag(cfg.D2_g)}_Dr{safe_tag(cfg.D2_r)}"


def g1_tag(D1_g, D1_r):
    return f"G1_Dg{safe_tag(D1_g)}_Dr{safe_tag(D1_r)}"


def game_tag(cfg, game):
    if cfg.model_type == "fixed":
        D_g, D_r = game
        return fixed_game_tag(D_g, D_r)
    else:
        D1_g, D1_r = game
        return os.path.join(g2_tag(cfg), g1_tag(D1_g, D1_r))


def base_parts(cfg):
    # Required order:
    # network → updating rule → async/sync → network randomicity → model → feedback → payoff
    parts = [
        network_tag(cfg),
        update_rule_tag(cfg),
        update_mode_tag(cfg),
        network_randomicity_tag(cfg),
        model_tag(cfg),
    ]

    fb = feedback_tag(cfg)
    if fb is not None:
        parts.append(fb)

    parts.append(payoff_mode_tag(cfg))

    # Add this only when self-interaction is active.
    # Therefore all previous folder names remain unchanged.
    self_tag = self_interaction_tag(cfg)
    if self_tag is not None:
        parts.append(self_tag)

    return parts


def make_rooted_paths(cfg, game=None, net_id=None):
    parts = base_parts(cfg)

    if game is not None:
        parts.append(game_tag(cfg, game))

    if net_id is not None:
        parts.append(f"ng_{int(net_id):03d}")

    root_tag = os.path.join(*parts)

    out_dir = os.path.join(cfg.out_root, root_tag)
    fig_dir = os.path.join(cfg.fig_root, root_tag)
    chk_dir = os.path.join(cfg.chk_root, root_tag)
    snap_dir = os.path.join(cfg.snap_root, root_tag)

    net_dir = os.path.join(
        cfg.net_root,
        os.path.join(*base_parts(cfg)),
        f"ng_{int(net_id or 0):03d}",
    )

    for d in (out_dir, fig_dir, chk_dir, snap_dir, net_dir):
        os.makedirs(d, exist_ok=True)

    return out_dir, fig_dir, chk_dir, snap_dir, net_dir


def results_paths(cfg, game, rho0, net_id, run):
    out_dir, *_ = make_rooted_paths(cfg, game=game, net_id=net_id)
    folder = os.path.join(out_dir, f"r0_{safe_tag(rho0)}_run_{run}")
    os.makedirs(folder, exist_ok=True)
    return folder, os.path.join(folder, "results.npz"), os.path.join(folder, "metadata.json")


def checkpoint_path(cfg, game, rho0, net_id, run, sweep=None):
    _, _, chk_dir, *_ = make_rooted_paths(cfg, game=game, net_id=net_id)
    mode = cfg.checkpoint_mode.lower()

    base = f"r0_{safe_tag(rho0)}_run_{run}"

    if mode == "overwrite":
        return os.path.join(chk_dir, base + ".npz")

    if mode == "history":
        if sweep is None:
            raise ValueError("In history checkpoint mode, sweep must be provided.")
        return os.path.join(chk_dir, base + f"_sw_{int(sweep):07d}.npz")

    raise ValueError("checkpoint_mode must be 'overwrite' or 'history'")


def latest_checkpoint_path(cfg, game, rho0, net_id, run):
    _, _, chk_dir, *_ = make_rooted_paths(cfg, game=game, net_id=net_id)
    mode = cfg.checkpoint_mode.lower()

    if mode == "overwrite":
        path = checkpoint_path(cfg, game, rho0, net_id, run)
        return path if os.path.exists(path) else None

    if mode == "history":
        prefix = f"r0_{safe_tag(rho0)}_run_{run}_sw_"
        suffix = ".npz"
        candidates = []

        for fname in os.listdir(chk_dir):
            if fname.startswith(prefix) and fname.endswith(suffix):
                m = re.search(r"_sw_(\d+)\.npz$", fname)
                if m:
                    candidates.append((int(m.group(1)), os.path.join(chk_dir, fname)))

        if not candidates:
            return None

        candidates.sort(key=lambda x: x[0])
        return candidates[-1][1]

    raise ValueError("checkpoint_mode must be 'overwrite' or 'history'")


def snapshot_dir(cfg, game, rho0, net_id, run):
    _, _, _, snap_root, _ = make_rooted_paths(cfg, game=game, net_id=net_id)
    folder = os.path.join(snap_root, f"r0_{safe_tag(rho0)}_run_{run}")
    os.makedirs(folder, exist_ok=True)
    return folder


def network_file_path(cfg, net_id):
    *_, net_dir = make_rooted_paths(cfg, game=None, net_id=net_id)
    return os.path.join(net_dir, "network_data.npz")


def layout_file_path(cfg, net_id):
    *_, net_dir = make_rooted_paths(cfg, game=None, net_id=net_id)
    return os.path.join(net_dir, "layout_positions.npz")


def degree_histogram_path(cfg, net_id):
    _, fig_dir, *_ = make_rooted_paths(cfg, game=None, net_id=net_id)
    return os.path.join(fig_dir, "deg_hist.png")


def degree_loglog_path(cfg, net_id):
    _, fig_dir, *_ = make_rooted_paths(cfg, game=None, net_id=net_id)
    return os.path.join(fig_dir, "deg_loglog.png")


def time_series_plot_path(cfg, game, rho0, net_id, run):
    _, fig_dir, *_ = make_rooted_paths(cfg, game=game, net_id=net_id)
    return os.path.join(fig_dir, f"rho_r0{safe_tag(rho0)}_run{run}.png")


def dgdr_series_plot_path(cfg, game, rho0, net_id, run):
    _, fig_dir, *_ = make_rooted_paths(cfg, game=game, net_id=net_id)
    return os.path.join(fig_dir, f"dgdr_r0{safe_tag(rho0)}_run{run}.png")


def early_stop_status(rho_series, current_sweep, nC, N_eff, cfg):
    """
    Returns:
        stop_now: bool
        reason: str or None
    """

    if not cfg.early_stop_enabled:
        return False, None

    # Exact absorbing states
    if cfg.early_stop_absorbing:
        if nC == 0:
            return True, "absorbing_all_defectors"
        if nC == N_eff:
            return True, "absorbing_all_cooperators"

    # Do not test too early
    if current_sweep < cfg.early_stop_min_sweeps:
        return False, None

    # Convert window from sweeps to number of recorded points
    records_needed = int(np.ceil(cfg.early_stop_window / cfg.record_every))
    records_needed = max(records_needed, 4)

    if len(rho_series) < records_needed:
        return False, None

    window = np.asarray(rho_series[-records_needed:], dtype=np.float64)

    # Criterion 1: total fluctuation in the window
    rho_range = float(np.max(window) - np.min(window))

    if rho_range > cfg.early_stop_rho_tol:
        return False, None

    # Criterion 2: no systematic drift between first and second half
    half = records_needed // 2
    first_half_mean = float(np.mean(window[:half]))
    second_half_mean = float(np.mean(window[half:]))

    slope_proxy = abs(second_half_mean - first_half_mean)

    if slope_proxy > cfg.early_stop_slope_tol:
        return False, None

    return True, (
        f"rho_stabilized: range={rho_range:.3e}, "
        f"slope_proxy={slope_proxy:.3e}"
    )


## Network Construction

In [ ]:
# ============================================================
# 3. NETWORK CONSTRUCTION
# ============================================================

def generate_ring_regular_lattice_graph(N, k):
    if k <= 0:
        raise ValueError("For ring_regular_lattice, k must be positive.")
    if k % 2 != 0:
        raise ValueError("For ring_regular_lattice, k must be even.")
    if k >= N:
        raise ValueError("For ring_regular_lattice, k must satisfy k < N.")

    G = nx.Graph()
    G.add_nodes_from(range(N))

    half = k // 2
    for i in range(N):
        for d in range(1, half + 1):
            G.add_edge(i, (i + d) % N)

    return G


def generate_square_lattice_graph(N, periodic=True, neighborhood="von_neumann"):
    """
    Generate a 2D square lattice.

    neighborhood:
        "von_neumann" -> 4 neighbours
        "moore"       -> 8 neighbours
    """

    L = int(round(np.sqrt(N)))

    if L * L != N:
        raise ValueError("For square_lattice, N must be a perfect square.")

    neighborhood = neighborhood.lower()

    if neighborhood not in ["von_neumann", "moore"]:
        raise ValueError("neighborhood must be 'von_neumann' or 'moore'.")

    G = nx.Graph()
    G.add_nodes_from(range(N))

    def node_id(i, j):
        return i * L + j

    if neighborhood == "von_neumann":
        directions = [
            (1, 0),
            (-1, 0),
            (0, 1),
            (0, -1),
        ]

    elif neighborhood == "moore":
        directions = [
            (1, 0),
            (-1, 0),
            (0, 1),
            (0, -1),
            (1, 1),
            (1, -1),
            (-1, 1),
            (-1, -1),
        ]

    for i in range(L):
        for j in range(L):
            u = node_id(i, j)

            for di, dj in directions:
                ni = i + di
                nj = j + dj

                if periodic:
                    ni %= L
                    nj %= L
                else:
                    if ni < 0 or ni >= L or nj < 0 or nj >= L:
                        continue

                v = node_id(ni, nj)

                if u != v:
                    G.add_edge(u, v)

    return G


def apply_degree_preserving_shuffles(G, alpha, ensure_connected=False, seed=None):
    if alpha is None:
        alpha = 0.0

    alpha = float(alpha)
    if alpha < 0:
        raise ValueError("alpha must be >= 0.")

    M = G.number_of_edges()
    n_swaps = int(round(alpha * M))

    if n_swaps == 0:
        return G.copy()

    G2 = G.copy()

    if ensure_connected:
        nx.connected_double_edge_swap(
            G2,
            nswap=n_swaps,
            _window_threshold=3,
            seed=seed,
        )
    else:
        max_tries = max(20 * n_swaps, 100)
        nx.double_edge_swap(
            G2,
            nswap=n_swaps,
            max_tries=max_tries,
            seed=seed,
        )

    return G2


def graph_seed(cfg, net_id):
    return int(cfg.seed + 1_000_000 * int(net_id))


def generate_graph(cfg, net_id=0):
    nt = cfg.network_type.lower()
    p = cfg.network_params
    seed = graph_seed(cfg, net_id)

    if nt == "erdos_renyi":
        G = nx.erdos_renyi_graph(cfg.N, p["p"], seed=seed)

    elif nt == "watts_strogatz":
        G = nx.watts_strogatz_graph(cfg.N, p["k"], p["beta"], seed=seed)

    elif nt == "newman_watts_strogatz":
        G = nx.newman_watts_strogatz_graph(cfg.N, p["k"], p["p"], seed=seed)

    elif nt == "barabasi_albert":
        G = nx.barabasi_albert_graph(cfg.N, p["m"], seed=seed)

    elif nt == "random_regular":
        G = nx.random_regular_graph(p["k"], cfg.N, seed=seed)

    elif nt == "ring_regular_lattice":
        G = generate_ring_regular_lattice_graph(cfg.N, int(p["k"]))
        G = apply_degree_preserving_shuffles(
            G,
            alpha=float(p.get("alpha", 0.0)),
            ensure_connected=cfg.ensure_connected,
            seed=seed,
        )

    elif nt == "square_lattice":
        G = generate_square_lattice_graph(
            cfg.N,
            periodic=bool(p.get("periodic", True)),
            neighborhood=p.get("neighborhood", "von_neumann"),
        )

        G = apply_degree_preserving_shuffles(
            G,
            alpha=float(p.get("alpha", 0.0)),
            ensure_connected=cfg.ensure_connected,
            seed=seed,
        )
        
    elif nt == "complete_graph":
        max_complete_n = int(p.get("max_complete_n", 5000))

        if cfg.N > max_complete_n:
            raise ValueError(
                f"complete_graph with N={cfg.N} is too large for the current "
                f"NetworkX/CSR implementation. "
                f"Use N <= {max_complete_n}, or implement an optimized well-mixed kernel."
            )

        G = nx.complete_graph(cfg.N)

    else:
        raise ValueError("Unsupported network_type.")

    # Remove any accidental self-loops generated by NetworkX or rewiring.
    # We add controlled self-interaction only below.
    G.remove_edges_from(nx.selfloop_edges(G))

    if cfg.ensure_connected and not nx.is_connected(G):
        giant = max(nx.connected_components(G), key=len)
        G = nx.convert_node_labels_to_integers(G.subgraph(giant).copy())

    # Controlled self-interaction:
    # each node plays against itself in the payoff calculation.
    # This is active only when cfg.self_interaction=True.
    if getattr(cfg, "self_interaction", False):
        G.add_edges_from((i, i) for i in G.nodes())

    return G


def graph_to_csr_arrays(G):
    N_eff = G.number_of_nodes()
    indptr = np.zeros(N_eff + 1, dtype=np.int32)

    rows = []
    count = 0

    for i in range(N_eff):
        nbrs = np.array(sorted(G.neighbors(i)), dtype=np.int32)
        rows.append(nbrs)
        count += len(nbrs)
        indptr[i + 1] = count

    indices = np.empty(count, dtype=np.int32)
    pos = 0

    for nbrs in rows:
        k = len(nbrs)
        indices[pos:pos + k] = nbrs
        pos += k

    degree = np.diff(indptr).astype(np.int32)
    return indptr, indices, degree


def save_network_data(cfg, net_id, indptr, indices, degree):
    np.savez_compressed(
        network_file_path(cfg, net_id),
        indptr=indptr,
        indices=indices,
        degree=degree,
    )


def load_network_data(cfg, net_id):
    with np.load(network_file_path(cfg, net_id)) as d:
        return d["indptr"].copy(), d["indices"].copy(), d["degree"].copy()


def prepare_network(cfg, net_id=0):
    path = network_file_path(cfg, net_id)

    if os.path.exists(path):
        indptr, indices, degree = load_network_data(cfg, net_id)
        return (
            indptr.astype(np.int32),
            indices.astype(np.int32),
            degree.astype(np.int32),
        )

    G = generate_graph(cfg, net_id=net_id)
    indptr, indices, degree = graph_to_csr_arrays(G)
    save_network_data(cfg, net_id, indptr, indices, degree)
    return indptr, indices, degree


def reconstruct_graph_from_csr(indptr, indices):
    N_eff = len(indptr) - 1
    G = nx.Graph()
    G.add_nodes_from(range(N_eff))

    for i in range(N_eff):
        for pos in range(indptr[i], indptr[i + 1]):
            j = int(indices[pos])
            if i < j:
                G.add_edge(i, j)

    return G


## Visualisation Helpers

In [ ]:
# ============================================================
# 4. VISUALISATION HELPERS
# ============================================================

def save_degree_histogram(cfg, net_id, degree):
    k_min = int(degree.min())
    k_max = int(degree.max())
    bins = np.arange(k_min - 0.5, k_max + 1.5, 1)

    plt.figure(figsize=(7, 4.5))
    plt.hist(degree, bins=bins, edgecolor="black")
    plt.xlabel("Degree k")
    plt.ylabel("Count")
    plt.title(f"Degree histogram | {cfg.network_type} | netgen={net_id}")
    plt.xlim(k_min - 0.5, k_max + 0.5)
    plt.tight_layout()
    plt.savefig(degree_histogram_path(cfg, net_id), dpi=200)
    plt.close()


def save_degree_loglog(cfg, net_id, degree):
    values, counts = np.unique(degree, return_counts=True)
    prob = counts / counts.sum()

    mask = values > 0
    values = values[mask]
    prob = prob[mask]

    plt.figure(figsize=(6.5, 4.5))
    plt.scatter(values, prob, s=18)
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("Degree k")
    plt.ylabel("P(k)")
    plt.title(f"Degree distribution | {cfg.network_type} | netgen={net_id}")
    plt.tight_layout()
    plt.savefig(degree_loglog_path(cfg, net_id), dpi=200)
    plt.close()


def get_or_create_layout(cfg, net_id, indptr, indices):
    path = layout_file_path(cfg, net_id)

    if os.path.exists(path):
        with np.load(path) as d:
            return d["pos"]

    G = reconstruct_graph_from_csr(indptr, indices)
    pos_dict = nx.spring_layout(G, seed=cfg.snapshot_layout_seed)

    pos = np.empty((len(pos_dict), 2), dtype=np.float64)
    for i in range(len(pos_dict)):
        pos[i, 0] = pos_dict[i][0]
        pos[i, 1] = pos_dict[i][1]

    np.savez_compressed(path, pos=pos)
    return pos


def save_network_snapshot(path, indptr, indices, state, pos, cfg):
    G = reconstruct_graph_from_csr(indptr, indices)
    pos_dict = {i: pos[i] for i in range(pos.shape[0])}

    colors = ["royalblue" if s == 0 else "crimson" for s in state]

    plt.figure(figsize=(8, 8))
    nx.draw_networkx_nodes(
        G,
        pos_dict,
        node_size=cfg.snapshot_node_size,
        node_color=colors,
        alpha=0.9,
    )
    nx.draw_networkx_edges(
        G,
        pos_dict,
        width=cfg.snapshot_edge_width,
        alpha=cfg.snapshot_edge_alpha,
    )
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()


def classify_game(Dg, Dr):
    if Dg > 0 and Dr > 0:
        return "PD"
    if Dg < 0 and Dr > 0:
        return "SH"
    if Dg > 0 and Dr < 0:
        return "SD"
    if Dg < 0 and Dr < 0:
        return "HG"
    return "boundary"


def plot_selected_games_in_dilemma_space(cfg):
    if cfg.model_type == "fixed":
        pairs = cfg.fixed_game_pairs
        Dg_vals = [g for g, r in pairs]
        Dr_vals = [r for g, r in pairs]
        title = "Selected fixed games in dilemma space"
    else:
        pairs = cfg.G1_pairs
        Dg_vals = [g for g, r in pairs]
        Dr_vals = [r for g, r in pairs]
        title = "Chosen G1 games and fixed G2 in dilemma space"

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.fill_between([0, 1], 0, 1, alpha=0.12)
    ax.fill_between([-1, 0], 0, 1, alpha=0.12)
    ax.fill_between([-1, 0], -1, 0, alpha=0.12)
    ax.fill_between([0, 1], -1, 0, alpha=0.12)

    vals = np.arange(-1.0, 1.0 + 0.2, 0.2)
    for v in vals:
        ax.axhline(v, linestyle="--", linewidth=0.5, alpha=0.35, color="gray")
        ax.axvline(v, linestyle="--", linewidth=0.5, alpha=0.35, color="gray")

    ax.axhline(0, linewidth=1.0, color="black")
    ax.axvline(0, linewidth=1.0, color="black")

    if cfg.model_type == "fixed":
        ax.scatter(Dg_vals, Dr_vals, s=80, marker="o", edgecolor="black", zorder=3)
    else:
        ax.scatter(Dg_vals, Dr_vals, s=120, marker="s", facecolor="white", edgecolor="black", label=r"$G_1$", zorder=3)
        ax.scatter([cfg.D2_g], [cfg.D2_r], s=180, marker="s", facecolor="purple", edgecolor="black", label=r"$G_2$", zorder=4)
        ax.legend()

    ax.text(0.5, 0.5, "PD", ha="center", va="center", fontsize=12)
    ax.text(-0.5, 0.5, "SH", ha="center", va="center", fontsize=12)
    ax.text(-0.5, -0.5, "HG", ha="center", va="center", fontsize=12)
    ax.text(0.5, -0.5, "SD", ha="center", va="center", fontsize=12)

    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-1.05, 1.05)
    ax.set_aspect("equal")
    ax.set_xlabel(r"$D_g = T - R$")
    ax.set_ylabel(r"$D_r = P - S$")
    ax.set_title(title)

    _, fig_dir, *_ = make_rooted_paths(cfg, game=None, net_id=0)
    save_path = os.path.join(fig_dir, "selected_games_dilemma_space.png")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


## Numba Simulation Kernels

In [ ]:
# ============================================================
# 5. NUMBA SIMULATION KERNELS
# ============================================================

@njit
def feedback_value_numba(rho, fb_flag):
    if fb_flag == 0:
        return rho
    return rho * rho


@njit
def init_state_numba(N_eff, rho0, seed):
    np.random.seed(seed)
    state = np.empty(N_eff, dtype=np.int8)
    for i in range(N_eff):
        state[i] = np.int8(0) if np.random.random() < rho0 else np.int8(1)
    return state


@njit
def count_cooperators_numba(state):
    nC = 0
    for i in range(state.shape[0]):
        if state[i] == 0:
            nC += 1
    return nC


@njit
def pair_payoff_numba(s_i, s_j, Dg, Dr):
    # C = 0, D = 1
    # C vs C = 1
    # C vs D = -Dr
    # D vs C = 1 + Dg
    # D vs D = 0
    if s_i == 0:
        if s_j == 0:
            return 1.0
        return -Dr
    else:
        if s_j == 0:
            return 1.0 + Dg
        return 0.0


@njit
def compute_site_payoff_numba(state, node, indptr, indices, degree, Dg, Dr, normalize_int):
    p = 0.0
    s = state[node]

    for pos in range(indptr[node], indptr[node + 1]):
        nb = indices[pos]
        p += pair_payoff_numba(s, state[nb], Dg, Dr)

    if normalize_int == 1 and degree[node] > 0:
        p /= degree[node]

    return p


@njit
def compute_all_payoffs_numba(state, indptr, indices, degree, Dg, Dr, normalize_int):
    N_eff = state.shape[0]
    payoff = np.empty(N_eff, dtype=np.float64)
    for i in range(N_eff):
        payoff[i] = compute_site_payoff_numba(
            state, i, indptr, indices, degree, Dg, Dr, normalize_int
        )
    return payoff


@njit
def effective_game_numba(model_flag, nC, N_eff, D1_g, D1_r, D2_g, D2_r, fb_flag):
    if model_flag == 0:
        return D1_g, D1_r

    rho = nC / N_eff
    a = feedback_value_numba(rho, fb_flag)

    eff_Dg = (1.0 - a) * D1_g + a * D2_g
    eff_Dr = (1.0 - a) * D1_r + a * D2_r

    return eff_Dg, eff_Dr


@njit
def payoff_difference_normalizer(Dg, Dr, kx, ky, normalize_int):
    # Original rule:
    # probability = (P_y - P_x) / [k_max * D_max]
    # where D_max = max(T, 1) - min(S, 0).
    #
    # Here T = 1 + Dg and S = -Dr.
    T = 1.0 + Dg
    S = -Dr

    Dmax = max(T, 1.0) - min(S, 0.0)

    if Dmax <= 0.0:
        return 1.0

    if normalize_int == 1:
        # If payoffs are averaged, the maximum payoff difference is Dmax,
        # not k_max * Dmax.
        return Dmax

    kmax = max(kx, ky)
    if kmax <= 0:
        return Dmax

    return kmax * Dmax


@njit
def adoption_decision_numba(
    pi_i,
    pi_j,
    Dg,
    Dr,
    k_i,
    k_j,
    update_rule_flag,
    omega,
    normalize_int,
):
    if update_rule_flag == 0:
        # Fermi rule
        prob = 1.0 / (1.0 + np.exp(-omega * (pi_j - pi_i)))
        return np.random.random() < prob

    # Proportional imitation rule
    if pi_j <= pi_i:
        return False

    denom = payoff_difference_normalizer(Dg, Dr, k_i, k_j, normalize_int)
    prob = (pi_j - pi_i) / denom

    if prob > 1.0:
        prob = 1.0

    return np.random.random() < prob


@njit
def one_sweep_async_numba(
    state,
    nC,
    indptr,
    indices,
    degree,
    model_flag,
    update_rule_flag,
    D1_g,
    D1_r,
    D2_g,
    D2_r,
    fb_flag,
    omega,
    normalize_int,
):
    N_eff = state.shape[0]

    for _ in range(N_eff):
        i = np.random.randint(0, N_eff)

        if degree[i] == 0:
            continue

        j = indices[indptr[i] + np.random.randint(0, degree[i])]

        # If self-interaction is active, the focal node may appear in its own
        # neighbour list for payoff purposes. We do not allow self-imitation.
        while j == i and degree[i] > 1:
            j = indices[indptr[i] + np.random.randint(0, degree[i])]

        if j == i:
            continue

        Dg, Dr = effective_game_numba(
            model_flag, nC, N_eff, D1_g, D1_r, D2_g, D2_r, fb_flag
        )

        pi_i = compute_site_payoff_numba(
            state, i, indptr, indices, degree, Dg, Dr, normalize_int
        )
        pi_j = compute_site_payoff_numba(
            state, j, indptr, indices, degree, Dg, Dr, normalize_int
        )

        if adoption_decision_numba(
            pi_i, pi_j, Dg, Dr, degree[i], degree[j],
            update_rule_flag, omega, normalize_int
        ):
            old_s = state[i]
            new_s = state[j]

            if old_s != new_s:
                state[i] = new_s
                if new_s == 0:
                    nC += 1
                else:
                    nC -= 1

    return nC


@njit
def one_sweep_sync_numba(
    state,
    nC,
    indptr,
    indices,
    degree,
    model_flag,
    update_rule_flag,
    D1_g,
    D1_r,
    D2_g,
    D2_r,
    fb_flag,
    omega,
    normalize_int,
):
    N_eff = state.shape[0]

    # In synchronous updating the game is evaluated once at the beginning
    # of the generation, using the current global rho.
    Dg, Dr = effective_game_numba(
        model_flag, nC, N_eff, D1_g, D1_r, D2_g, D2_r, fb_flag
    )

    payoff = compute_all_payoffs_numba(
        state, indptr, indices, degree, Dg, Dr, normalize_int
    )

    old_state = state.copy()
    new_state = state.copy()

    for i in range(N_eff):
        if degree[i] == 0:
            continue

        j = indices[indptr[i] + np.random.randint(0, degree[i])]

        # If self-interaction is active, avoid self-imitation.
        while j == i and degree[i] > 1:
            j = indices[indptr[i] + np.random.randint(0, degree[i])]

        if j == i:
            continue

        if adoption_decision_numba(
            payoff[i], payoff[j], Dg, Dr, degree[i], degree[j],
            update_rule_flag, omega, normalize_int
        ):
            new_state[i] = old_state[j]

    nC_new = 0
    for i in range(N_eff):
        state[i] = new_state[i]
        if state[i] == 0:
            nC_new += 1

    return nC_new


@njit
def run_sweeps_numba(
    state,
    nC,
    indptr,
    indices,
    degree,
    model_flag,
    update_rule_flag,
    update_mode_flag,
    D1_g,
    D1_r,
    D2_g,
    D2_r,
    fb_flag,
    omega,
    normalize_int,
    seed,
    n_sweeps,
    record_every,
):
    np.random.seed(seed)

    N_eff = state.shape[0]
    n_records = n_sweeps // record_every + 1

    rho_records = np.empty(n_records, dtype=np.float64)
    Dg_records = np.empty(n_records, dtype=np.float64)
    Dr_records = np.empty(n_records, dtype=np.float64)

    rec_idx = 0

    Dg, Dr = effective_game_numba(
        model_flag, nC, N_eff, D1_g, D1_r, D2_g, D2_r, fb_flag
    )

    rho_records[rec_idx] = nC / N_eff
    Dg_records[rec_idx] = Dg
    Dr_records[rec_idx] = Dr
    rec_idx += 1

    for sweep_local in range(1, n_sweeps + 1):

        if update_mode_flag == 0:
            nC = one_sweep_async_numba(
                state,
                nC,
                indptr,
                indices,
                degree,
                model_flag,
                update_rule_flag,
                D1_g,
                D1_r,
                D2_g,
                D2_r,
                fb_flag,
                omega,
                normalize_int,
            )
        else:
            nC = one_sweep_sync_numba(
                state,
                nC,
                indptr,
                indices,
                degree,
                model_flag,
                update_rule_flag,
                D1_g,
                D1_r,
                D2_g,
                D2_r,
                fb_flag,
                omega,
                normalize_int,
            )

        if sweep_local % record_every == 0:
            Dg, Dr = effective_game_numba(
                model_flag, nC, N_eff, D1_g, D1_r, D2_g, D2_r, fb_flag
            )

            rho_records[rec_idx] = nC / N_eff
            Dg_records[rec_idx] = Dg
            Dr_records[rec_idx] = Dr
            rec_idx += 1

    return nC, rho_records[:rec_idx], Dg_records[:rec_idx], Dr_records[:rec_idx]

@njit
def sampled_payoff_well_mixed_numba(
    state,
    player,
    k_payoff,
    Dg,
    Dr,
    normalize_int,
):
    """
    Payoff of one player in a well-mixed sampled population.

    The player interacts with k_payoff randomly sampled opponents.
    Opponents are sampled from the population excluding the player itself.

    If normalize_int == 0:
        total payoff over k_payoff interactions.

    If normalize_int == 1:
        average payoff over k_payoff interactions.
    """

    N_eff = state.shape[0]
    s_player = state[player]

    payoff = 0.0

    for _ in range(k_payoff):
        opponent = np.random.randint(0, N_eff)

        while opponent == player:
            opponent = np.random.randint(0, N_eff)

        payoff += pair_payoff_numba(
            s_player,
            state[opponent],
            Dg,
            Dr,
        )

    if normalize_int == 1 and k_payoff > 0:
        payoff /= k_payoff

    return payoff

@njit
def one_sweep_well_mixed_sampled_numba(
    state,
    nC,
    model_flag,
    update_rule_flag,
    D1_g,
    D1_r,
    D2_g,
    D2_r,
    fb_flag,
    omega,
    normalize_int,
    k_payoff,
):
    """
    One Monte Carlo sweep for the sampled well-mixed protocol.

    One sweep = N elementary update attempts.

    At each elementary update:
    - choose focal player f randomly;
    - choose model player m randomly;
    - compute payoff of f against k_payoff random opponents;
    - compute payoff of m against k_payoff random opponents;
    - update f by Fermi/proportional imitation.
    """

    N_eff = state.shape[0]

    for _ in range(N_eff):
        f = np.random.randint(0, N_eff)
        m = np.random.randint(0, N_eff)

        while m == f:
            m = np.random.randint(0, N_eff)

        Dg, Dr = effective_game_numba(
            model_flag,
            nC,
            N_eff,
            D1_g,
            D1_r,
            D2_g,
            D2_r,
            fb_flag,
        )

        pi_f = sampled_payoff_well_mixed_numba(
            state,
            f,
            k_payoff,
            Dg,
            Dr,
            normalize_int,
        )

        pi_m = sampled_payoff_well_mixed_numba(
            state,
            m,
            k_payoff,
            Dg,
            Dr,
            normalize_int,
        )

        if adoption_decision_numba(
            pi_f,
            pi_m,
            Dg,
            Dr,
            k_payoff,
            k_payoff,
            update_rule_flag,
            omega,
            normalize_int,
        ):
            old_s = state[f]
            new_s = state[m]

            if old_s != new_s:
                state[f] = new_s

                if new_s == 0:
                    nC += 1
                else:
                    nC -= 1

    return nC

@njit
def run_sweeps_well_mixed_sampled_numba(
    state,
    nC,
    model_flag,
    update_rule_flag,
    D1_g,
    D1_r,
    D2_g,
    D2_r,
    fb_flag,
    omega,
    normalize_int,
    seed,
    n_sweeps,
    record_every,
    k_payoff,
):
    np.random.seed(seed)

    N_eff = state.shape[0]
    n_records = n_sweeps // record_every + 1

    rho_records = np.empty(n_records, dtype=np.float64)
    Dg_records = np.empty(n_records, dtype=np.float64)
    Dr_records = np.empty(n_records, dtype=np.float64)

    rec_idx = 0

    Dg, Dr = effective_game_numba(
        model_flag,
        nC,
        N_eff,
        D1_g,
        D1_r,
        D2_g,
        D2_r,
        fb_flag,
    )

    rho_records[rec_idx] = nC / N_eff
    Dg_records[rec_idx] = Dg
    Dr_records[rec_idx] = Dr
    rec_idx += 1

    for sweep_local in range(1, n_sweeps + 1):
        nC = one_sweep_well_mixed_sampled_numba(
            state,
            nC,
            model_flag,
            update_rule_flag,
            D1_g,
            D1_r,
            D2_g,
            D2_r,
            fb_flag,
            omega,
            normalize_int,
            k_payoff,
        )

        if sweep_local % record_every == 0:
            Dg, Dr = effective_game_numba(
                model_flag,
                nC,
                N_eff,
                D1_g,
                D1_r,
                D2_g,
                D2_r,
                fb_flag,
            )

            rho_records[rec_idx] = nC / N_eff
            Dg_records[rec_idx] = Dg
            Dr_records[rec_idx] = Dr
            rec_idx += 1

    return nC, rho_records[:rec_idx], Dg_records[:rec_idx], Dr_records[:rec_idx]

## Checkpoint and Result IO

In [ ]:
# ============================================================
# 6. CHECKPOINT AND RESULT IO
# ============================================================

def save_checkpoint(path, state, nC, sweep, py_rng_state, max_retries=10, delay=0.2):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp.npz"

    np.savez_compressed(
        tmp,
        state=state,
        nC=np.array([int(nC)]),
        sweep=np.array([int(sweep)]),
        rng_state=np.array([py_rng_state], dtype=object),
    )

    gc.collect()

    last_err = None
    for _ in range(max_retries):
        try:
            os.replace(tmp, path)
            return
        except PermissionError as e:
            last_err = e
            time.sleep(delay)

    raise last_err


def load_checkpoint(path):
    with np.load(path, allow_pickle=True) as chk:
        state = chk["state"].copy()
        nC = int(chk["nC"][0])
        sweep = int(chk["sweep"][0])
        py_rng_state = chk["rng_state"][0]

    return state, nC, sweep, py_rng_state


def append_series(save_npz, save_meta, rho_new, Dg_new, Dr_new, meta_update, start_fresh=False):
    if (not start_fresh) and os.path.exists(save_npz):
        with np.load(save_npz, allow_pickle=True) as old:
            rho_all = np.concatenate([old["rho_series"], rho_new])
            Dg_all = np.concatenate([old["Dg_series"], Dg_new])
            Dr_all = np.concatenate([old["Dr_series"], Dr_new])
    else:
        rho_all = rho_new
        Dg_all = Dg_new
        Dr_all = Dr_new

    tmp = save_npz + ".tmp.npz"
    np.savez_compressed(
        tmp,
        rho_series=rho_all,
        Dg_series=Dg_all,
        Dr_series=Dr_all,
    )
    os.replace(tmp, save_npz)

    meta = meta_update.copy()
    meta["recorded_points_total"] = int(len(rho_all))

    with open(save_meta, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, sort_keys=True)


## High-Level Simulation Functions

In [ ]:
# ============================================================
# 7. HIGH-LEVEL SIMULATION FUNCTIONS
# ============================================================

def model_flag(cfg):
    if cfg.model_type == "fixed":
        return 0
    if cfg.model_type == "endogenous":
        return 1
    raise ValueError("model_type must be 'fixed' or 'endogenous'")


def update_rule_flag(cfg):
    if cfg.update_rule == "fermi":
        return 0
    if cfg.update_rule == "proportional_imitation":
        return 1
    raise ValueError("update_rule must be 'fermi' or 'proportional_imitation'")


def update_mode_flag(cfg):
    if cfg.update_mode == "async":
        return 0
    if cfg.update_mode == "sync":
        return 1
    raise ValueError("update_mode must be 'async' or 'sync'")


def feedback_flag(cfg):
    if cfg.feedback == "rho":
        return 0
    if cfg.feedback == "rho2":
        return 1
    raise ValueError("feedback must be 'rho' or 'rho2'")


def effective_dg_dr_python(rho, D1_g, D1_r, D2_g, D2_r, feedback):
    a = rho if feedback == "rho" else rho * rho
    return (1.0 - a) * D1_g + a * D2_g, (1.0 - a) * D1_r + a * D2_r


def unpack_game_for_kernel(cfg, game):
    if cfg.model_type == "fixed":
        D_g, D_r = game
        return float(D_g), float(D_r), 0.0, 0.0
    else:
        D1_g, D1_r = game
        return float(D1_g), float(D1_r), float(cfg.D2_g), float(cfg.D2_r)


def task_seed(cfg, game, rho0, net_id, run):
    D1_g, D1_r, D2_g, D2_r = unpack_game_for_kernel(cfg, game)

    return int(
        cfg.seed
        + 1_000_000 * int(net_id)
        + 100_000 * int(run)
        + 10_000 * int(round(rho0 * 1000))
        + 13 * int(round((D1_g + 2.0) * 1000))
        + 29 * int(round((D1_r + 2.0) * 1000))
        + 17 * int(round((D2_g + 2.0) * 1000))
        + 31 * int(round((D2_r + 2.0) * 1000))
    )


def run_one_task(task, cfg_dict):
    game, rho0, net_id, run = task
    local_cfg = Config(**cfg_dict)

    folder, save_npz, save_meta = results_paths(local_cfg, game, rho0, net_id, run)
    snap_folder = snapshot_dir(local_cfg, game, rho0, net_id, run)

    is_well_mixed_sampled = local_cfg.network_type.lower() == "well_mixed_sampled"

    if is_well_mixed_sampled:
        indptr = None
        indices = None
        degree = None
        N_eff = int(local_cfg.N)
        pos = None
    else:
        indptr, indices, degree = prepare_network(local_cfg, net_id=net_id)
        N_eff = len(degree)

        pos = None
        if (
            local_cfg.save_snapshots
            or local_cfg.save_initial_snapshot
            or local_cfg.save_final_snapshot
            or local_cfg.save_network_visualization
        ):
            pos = get_or_create_layout(local_cfg, net_id, indptr, indices)

    py_rng = np.random.default_rng(task_seed(local_cfg, game, rho0, net_id, run))

    resumed = False
    resume_chk_path = None

    if local_cfg.save_checkpoints:
        resume_chk_path = latest_checkpoint_path(local_cfg, game, rho0, net_id, run)

    if local_cfg.save_checkpoints and resume_chk_path is not None:
        state, nC, start_sweep, saved_rng_state = load_checkpoint(resume_chk_path)
        py_rng.bit_generator.state = saved_rng_state
        resumed = True
    else:
        init_seed = uint32(py_rng.integers(0, 2**32 - 1))
        state = init_state_numba(N_eff, float(rho0), init_seed)
        nC = int(count_cooperators_numba(state))
        start_sweep = 0

        if local_cfg.save_checkpoints:
            init_chk = checkpoint_path(local_cfg, game, rho0, net_id, run, sweep=0)
            save_checkpoint(
                init_chk,
                state,
                nC,
                start_sweep,
                py_rng.bit_generator.state,
            )

    requested_end_sweep = start_sweep + int(local_cfg.extra_sweeps)
    end_sweep = requested_end_sweep

    if local_cfg.save_initial_snapshot and pos is not None and not resumed:
        snap_file = os.path.join(snap_folder, f"snap_{start_sweep:07d}.png")
        if not os.path.exists(snap_file):
            save_network_snapshot(snap_file, indptr, indices, state, pos, local_cfg)

    checkpoint_sweeps = set()
    snapshot_sweeps = set()
    early_stop_sweeps = set()

    if local_cfg.save_checkpoints:
        for s in range(start_sweep + 1, end_sweep + 1):
            if s % local_cfg.checkpoint_every == 0:
                checkpoint_sweeps.add(s)

    if local_cfg.save_snapshots:
        for s in range(start_sweep + 1, end_sweep + 1):
            if s % local_cfg.snap_every == 0:
                snapshot_sweeps.add(s)

    if local_cfg.early_stop_enabled:
        check_every = int(local_cfg.early_stop_check_every)

        if check_every <= 0:
            raise ValueError("early_stop_check_every must be positive.")

        first_check = ((start_sweep // check_every) + 1) * check_every

        for s in range(first_check, end_sweep + 1, check_every):
            early_stop_sweeps.add(s)

    boundary_sweeps = sorted(
        checkpoint_sweeps | snapshot_sweeps | early_stop_sweeps | {end_sweep}
    )

    rho_records_all = []
    Dg_records_all = []
    Dr_records_all = []

    D1_g, D1_r, D2_g, D2_r = unpack_game_for_kernel(local_cfg, game)
    rho_now = nC / N_eff

    if local_cfg.model_type == "fixed":
        Dg_now, Dr_now = D1_g, D1_r
    else:
        Dg_now, Dr_now = effective_dg_dr_python(
            rho_now,
            D1_g,
            D1_r,
            D2_g,
            D2_r,
            local_cfg.feedback,
        )

    rho_records_all.append(rho_now)
    Dg_records_all.append(Dg_now)
    Dr_records_all.append(Dr_now)

    current_sweep = start_sweep
    saved_checkpoint_paths = []

    stopped_early = False
    early_stop_reason = None

    # Check immediately in case the loaded/initial state is already absorbing.
    stop_now, stop_reason = early_stop_status(
        rho_series=rho_records_all,
        current_sweep=current_sweep,
        nC=nC,
        N_eff=N_eff,
        cfg=local_cfg,
    )

    if stop_now:
        stopped_early = True
        early_stop_reason = stop_reason

    if not stopped_early:
        for boundary in boundary_sweeps:
            chunk_size = boundary - current_sweep

            if chunk_size <= 0:
                continue

            chunk_seed = uint32(py_rng.integers(0, 2**32 - 1))

            if is_well_mixed_sampled:
                k_payoff = int(local_cfg.network_params.get("k_payoff", 4))

                nC, rho_chunk, Dg_chunk, Dr_chunk = run_sweeps_well_mixed_sampled_numba(
                    state=state,
                    nC=nC,
                    model_flag=model_flag(local_cfg),
                    update_rule_flag=update_rule_flag(local_cfg),
                    D1_g=D1_g,
                    D1_r=D1_r,
                    D2_g=D2_g,
                    D2_r=D2_r,
                    fb_flag=feedback_flag(local_cfg),
                    omega=float(local_cfg.omega),
                    normalize_int=1 if local_cfg.normalize_payoff else 0,
                    seed=chunk_seed,
                    n_sweeps=chunk_size,
                    record_every=local_cfg.record_every,
                    k_payoff=k_payoff,
                )

            else:
                nC, rho_chunk, Dg_chunk, Dr_chunk = run_sweeps_numba(
                    state=state,
                    nC=nC,
                    indptr=indptr,
                    indices=indices,
                    degree=degree,
                    model_flag=model_flag(local_cfg),
                    update_rule_flag=update_rule_flag(local_cfg),
                    update_mode_flag=update_mode_flag(local_cfg),
                    D1_g=D1_g,
                    D1_r=D1_r,
                    D2_g=D2_g,
                    D2_r=D2_r,
                    fb_flag=feedback_flag(local_cfg),
                    omega=float(local_cfg.omega),
                    normalize_int=1 if local_cfg.normalize_payoff else 0,
                    seed=chunk_seed,
                    n_sweeps=chunk_size,
                    record_every=local_cfg.record_every,
                )

            rho_records_all.extend(rho_chunk[1:].tolist())
            Dg_records_all.extend(Dg_chunk[1:].tolist())
            Dr_records_all.extend(Dr_chunk[1:].tolist())

            current_sweep = boundary

            stop_now, stop_reason = early_stop_status(
                rho_series=rho_records_all,
                current_sweep=current_sweep,
                nC=nC,
                N_eff=N_eff,
                cfg=local_cfg,
            )

            if stop_now:
                stopped_early = True
                early_stop_reason = stop_reason
                break

            if boundary in snapshot_sweeps and pos is not None:
                snap_file = os.path.join(snap_folder, f"snap_{boundary:07d}.png")
                save_network_snapshot(
                    snap_file,
                    indptr,
                    indices,
                    state,
                    pos,
                    local_cfg,
                )

            if boundary in checkpoint_sweeps:
                chk_save_path = checkpoint_path(
                    local_cfg,
                    game,
                    rho0,
                    net_id,
                    run,
                    sweep=boundary,
                )

                save_checkpoint(
                    chk_save_path,
                    state,
                    nC,
                    boundary,
                    py_rng.bit_generator.state,
                )

                saved_checkpoint_paths.append(chk_save_path)

    actual_end_sweep = current_sweep

    if local_cfg.save_final_snapshot and pos is not None:
        final_snap = os.path.join(
            snap_folder,
            f"snap_{actual_end_sweep:07d}.png",
        )

        if not os.path.exists(final_snap):
            save_network_snapshot(
                final_snap,
                indptr,
                indices,
                state,
                pos,
                local_cfg,
            )

    final_chk_path = None

    if local_cfg.save_checkpoints and local_cfg.save_final_checkpoint:
        final_chk_path = checkpoint_path(
            local_cfg,
            game,
            rho0,
            net_id,
            run,
            sweep=actual_end_sweep,
        )

        if final_chk_path not in saved_checkpoint_paths:
            save_checkpoint(
                final_chk_path,
                state,
                nC,
                actual_end_sweep,
                py_rng.bit_generator.state,
            )

            saved_checkpoint_paths.append(final_chk_path)

    rho_chunk_np = np.array(rho_records_all, dtype=np.float64)
    Dg_chunk_np = np.array(Dg_records_all, dtype=np.float64)
    Dr_chunk_np = np.array(Dr_records_all, dtype=np.float64)

    if resumed:
        rho_chunk_np = rho_chunk_np[1:]
        Dg_chunk_np = Dg_chunk_np[1:]
        Dr_chunk_np = Dr_chunk_np[1:]

    start_fresh = (not resumed) and os.path.exists(save_npz)

    meta_update = {
        "model": "Unified EGT network simulator",
        "config": asdict(local_cfg),
        "model_type": local_cfg.model_type,
        "update_rule": local_cfg.update_rule,
        "update_mode": local_cfg.update_mode,
        "network_type": local_cfg.network_type,
        "network_params": local_cfg.network_params,
        "network_randomicity": {
            "randomize_network": bool(local_cfg.randomize_network),
            "network_generations": int(local_cfg.network_generations),
            "net_id": int(net_id),
        },
        "payoff_mode": payoff_mode_tag(local_cfg),
        "effective_N": int(N_eff),
        "game": {
            "raw_game_tuple": tuple(float(x) for x in game),
            "D1_g_or_D_g": float(D1_g),
            "D1_r_or_D_r": float(D1_r),
            "D2_g": float(D2_g),
            "D2_r": float(D2_r),
            "feedback": local_cfg.feedback
            if local_cfg.model_type == "endogenous"
            else None,
        },
        "rho0": float(rho0),
        "run": int(run),
        "results_folder": folder,
        "snapshot_dir": snap_folder,
        "resumed": bool(resumed),
        "checkpoint_mode": local_cfg.checkpoint_mode,
        "latest_checkpoint_used_for_resume": resume_chk_path,
        "saved_checkpoint_paths": saved_checkpoint_paths,
        "start_sweep": int(start_sweep),
        "requested_end_sweep": int(requested_end_sweep),
        "end_sweep": int(actual_end_sweep),
        "stopped_early": bool(stopped_early),
        "early_stop_reason": early_stop_reason,
        "note_on_proportional_imitation": (
            "With update_mode='sync', this follows the synchronous structure of the paper. "
            "With update_mode='async', it uses the same normalized probability but updates sequentially within sweeps."
        )
        if local_cfg.update_rule == "proportional_imitation"
        else None,
    }

    append_series(
        save_npz=save_npz,
        save_meta=save_meta,
        rho_new=rho_chunk_np,
        Dg_new=Dg_chunk_np,
        Dr_new=Dr_chunk_np,
        meta_update=meta_update,
        start_fresh=start_fresh,
    )

    return {
        "task": task,
        "resumed": resumed,
        "start_sweep": start_sweep,
        "requested_end_sweep": requested_end_sweep,
        "end_sweep": actual_end_sweep,
        "stopped_early": stopped_early,
        "early_stop_reason": early_stop_reason,
        "results_file": save_npz,
        "checkpoint_mode": local_cfg.checkpoint_mode,
        "latest_checkpoint_used_for_resume": resume_chk_path,
        "final_checkpoint_file": final_chk_path,
        "snapshot_dir": snap_folder,
        "final_rho": float(nC / N_eff),
    }

## Main Simulation Launcher

In [ ]:
# ============================================================
# 8. MAIN SIMULATION LAUNCHER
# ============================================================

def games_to_run(cfg):
    if cfg.model_type == "fixed":
        return list(cfg.fixed_game_pairs)
    if cfg.model_type == "endogenous":
        return list(cfg.G1_pairs)
    raise ValueError("model_type must be 'fixed' or 'endogenous'")


def network_ids_to_run(cfg):
    if cfg.randomize_network:
        return list(range(int(cfg.network_generations)))
    return [0]


def prepare_all_networks_and_basic_figures(cfg):
    if cfg.network_type.lower() == "well_mixed_sampled":
        return

    for net_id in network_ids_to_run(cfg):
        indptr, indices, degree = prepare_network(cfg, net_id=net_id)
        save_degree_histogram(cfg, net_id, degree)
        save_degree_loglog(cfg, net_id, degree)

        if (
            cfg.save_snapshots
            or cfg.save_initial_snapshot
            or cfg.save_final_snapshot
            or cfg.save_network_visualization
        ):
            _ = get_or_create_layout(cfg, net_id, indptr, indices)


def main():
    print("====================================================")
    print("Unified EGT network simulator")
    print(f"Model type          : {cfg.model_type}")
    print(f"Update rule         : {cfg.update_rule}")
    print(f"Update mode         : {cfg.update_mode}")
    print(f"Network type        : {cfg.network_type}")
    print(f"Network params      : {cfg.network_params}")
    print(f"N                   : {cfg.N}")
    print(f"Network randomicity : {cfg.randomize_network}")
    print(f"Network generations : {cfg.network_generations}")
    print(f"Runs per network    : {cfg.runs_per_network}")
    print(f"Payoff mode         : {payoff_mode_tag(cfg)}")
    print(f"omega               : {cfg.omega}")
    print(f"extra_sweeps        : {cfg.extra_sweeps}")
    print(f"record_every        : {cfg.record_every}")
    print(f"rho0_values         : {cfg.rho0_values}")
    if cfg.model_type == "endogenous":
        print(f"Feedback            : {cfg.feedback}")
        print(f"G2                  : ({cfg.D2_g}, {cfg.D2_r})")
    print("====================================================")

    prepare_all_networks_and_basic_figures(cfg)
    plot_selected_games_in_dilemma_space(cfg)

    tasks = []
    for net_id in network_ids_to_run(cfg):
        for game in games_to_run(cfg):
            for rho0 in cfg.rho0_values:
                for run in range(cfg.runs_per_network):
                    tasks.append((game, rho0, net_id, run))

    n_workers = cfg.max_workers if cfg.max_workers is not None else os.cpu_count()
    cfg_dict = asdict(cfg)

    results = []

    if n_workers == 1:
        for task in tqdm(tasks, desc="Overall progress"):
            results.append(run_one_task(task, cfg_dict))
    else:
        with ProcessPoolExecutor(max_workers=n_workers) as ex:
            futures = [ex.submit(run_one_task, task, cfg_dict) for task in tasks]
            for fut in tqdm(as_completed(futures), total=len(futures), desc="Overall progress"):
                results.append(fut.result())

    print("\nAll tasks completed.\n")

    for r in results:
        game, rho0, net_id, run = r["task"]
        print(
            f"game={game}, netgen={net_id}, rho0={rho0}, run={run} | "
            f"resumed={r['resumed']} | "
            f"sweeps {r['start_sweep']} -> {r['end_sweep']} "
            f"(requested {r['requested_end_sweep']}) | "
            f"stopped_early={r['stopped_early']} | "
            f"final rho_C={r['final_rho']:.6f}"
)


# Uncomment to run:
main()


Unified EGT network simulator
Model type          : endogenous
Update rule         : fermi
Update mode         : async
Network type        : well_mixed_sampled
Network params      : {'k_payoff': 4}
N                   : 1000
Network randomicity : False
Network generations : 1
Runs per network    : 10
Payoff mode         : tot
omega               : 0.2
extra_sweeps        : 0
record_every        : 1
rho0_values         : (0.1, 0.5, 0.9)
Feedback            : rho
G2                  : (1.0, -1.0)


Overall progress:   0%|          | 0/120 [00:00<?, ?it/s]


All tasks completed.

game=(-0.6, -0.4), netgen=0, rho0=0.1, run=0 | resumed=False | sweeps 0 -> 0 (requested 0) | stopped_early=False | final rho_C=0.095000
game=(-0.6, -0.4), netgen=0, rho0=0.1, run=1 | resumed=False | sweeps 0 -> 0 (requested 0) | stopped_early=False | final rho_C=0.110000
game=(-0.6, -0.4), netgen=0, rho0=0.1, run=2 | resumed=False | sweeps 0 -> 0 (requested 0) | stopped_early=False | final rho_C=0.103000
game=(-0.6, -0.4), netgen=0, rho0=0.1, run=3 | resumed=False | sweeps 0 -> 0 (requested 0) | stopped_early=False | final rho_C=0.088000
game=(-0.6, -0.4), netgen=0, rho0=0.1, run=4 | resumed=False | sweeps 0 -> 0 (requested 0) | stopped_early=False | final rho_C=0.102000
game=(-0.6, -0.4), netgen=0, rho0=0.1, run=5 | resumed=False | sweeps 0 -> 0 (requested 0) | stopped_early=False | final rho_C=0.096000
game=(-0.6, -0.4), netgen=0, rho0=0.1, run=6 | resumed=False | sweeps 0 -> 0 (requested 0) | stopped_early=False | final rho_C=0.112000
game=(-0.6, -0.4), netgen